# Logistic Regression: Estratégias Multiclasse

Análise comparativa das diferentes estratégias de classificação multiclasse
disponíveis no `LogisticRegression` do scikit-learn:

- **Multinomial (softmax nativo)**: uma única matriz de coeficientes, probabilidades
  somam 1 via função softmax. Pressupõe que as classes são mutuamente exclusivas.
- **One-vs-Rest (OvR)**: um classificador binário por classe (cada classe vs.
  todas as outras). Probabilidades são calibradas independentemente.
- **One-vs-One (OvO)**: um classificador binário para cada par de classes.
  Predição final via votação entre os pares.

Dataset: `jp797498e/twitter-entity-sentiment-analysis` (4 classes: Positive,
Negative, Neutral, Irrelevant).

In [1]:
# ─── Imports ────────────────────────────────────────────────────
import kagglehub
import pandas as pd
import numpy as np
import re
import time
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsOneClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, f1_score
)

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

SEED = 42
np.random.seed(SEED)

print('Imports OK \u2714')

Imports OK ✔


## 1. Carregamento e Pré-processamento

In [2]:
# ─── Carrega dados via kagglehub ────────────────────────────────────────
path = kagglehub.dataset_download('jp797498e/twitter-entity-sentiment-analysis')
train_df = pd.read_csv(os.path.join(path, 'twitter_training.csv'))
val_df = pd.read_csv(os.path.join(path, 'twitter_validation.csv'))

print(f'Treino : {train_df.shape}')
print(f'Valida\u00e7\u00e3o: {val_df.shape}')

Treino : (74681, 4)
Validação: (999, 4)


In [3]:
# ─── Padroniza nomes de colunas ─────────────────────────────────────────────
COL_NAMES = ['id', 'entity', 'sentiment', 'text']

if train_df.shape[1] == 4:
    train_df.columns = COL_NAMES
    val_df.columns   = COL_NAMES

print('Sentimentos \u00fanicos:', train_df['sentiment'].unique())
print('Distribui\u00e7\u00e3o no treino:')
print(train_df['sentiment'].value_counts())

Sentimentos únicos: ['Positive' 'Negative' 'Neutral' 'Irrelevant']
Distribuição no treino:
sentiment
Negative      22542
Positive      20831
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64


In [4]:
# ─── Função de limpeza ───────────────────────────────────────────────
def clean_tweet(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[^a-z0-9\s!?.,\'\-]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

for df in (train_df, val_df):
    df['clean_text'] = df['text'].apply(clean_tweet)

valid_sentiments = ['Positive', 'Negative', 'Neutral', 'Irrelevant']
train_df = train_df[
    train_df['clean_text'].str.len() > 0 &
    train_df['sentiment'].isin(valid_sentiments)
].copy()

val_df = val_df[
    val_df['clean_text'].str.len() > 0 &
    val_df['sentiment'].isin(valid_sentiments)
].copy()

print(f'Treino ap\u00f3s limpeza : {train_df.shape}')
print(f'Valida\u00e7\u00e3o ap\u00f3s limpeza: {val_df.shape}')

Treino após limpeza : (73768, 6)
Validação após limpeza: (999, 6)


In [5]:
# ─── Encode labels ─────────────────────────────────────────────────
le = LabelEncoder()
le.fit(valid_sentiments)

y_train = le.transform(train_df['sentiment'])
y_val   = le.transform(val_df['sentiment'])

X_train_text = train_df['clean_text'].values
X_val_text   = val_df['clean_text'].values

print('Classes:', dict(zip(le.classes_, le.transform(le.classes_))))

Classes: {'Irrelevant': 0, 'Negative': 1, 'Neutral': 2, 'Positive': 3}


## 2. TF-IDF Vectorization

In [6]:
# ─── TF-IDF Vectorizer ──────────────────────────────────────────────
tfidf = TfidfVectorizer(
    max_features=70_000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    strip_accents='unicode',
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_val_tfidf   = tfidf.transform(X_val_text)

print(f'Shape TF-IDF treino : {X_train_tfidf.shape}')
print(f'Shape TF-IDF valida\u00e7\u00e3o: {X_val_tfidf.shape}')

Shape TF-IDF treino : (73768, 70000)
Shape TF-IDF validação: (999, 70000)


## 3. Experimentos com Multi-Class

Testamos 5 configurações que abrangem todas as estratégias
multiclasse disponíveis no scikit-learn:

| # | Estratégia | `multi_class` | `solver` | Descrição |
|---|--------------|---------------|----------|-------------------|
| 1 | Multinomial  | `multinomial` | `lbfgs`   | Softmax nativo (uma matriz de pesos W \u00d7 K) |
| 2 | OvR (lbfgs)  | `ovr`         | `lbfgs`   | One-vs-Rest com solver quasi-Newton |
| 3 | OvR (liblinear) | `ovr`     | `liblinear` | One-vs-Rest com coordenada descendente |
| 4 | OvR (saga)   | `ovr`         | `saga`    | One-vs-Rest com gradiente estocástico |
| 5 | OvO (liblinear) | (wrap)     | `liblinear` | One-vs-One via `OneVsOneClassifier` |

Cada configuração é avaliada em 4 valores de C (regularização inversa).

In [7]:
# ─── Configurações de experimento ──────────────────────────────────────
C_VALUES = [0.1, 1.0, 10.0, 100.0]

STRATEGIES = [
    ('Multinomial (lbfgs)', 'multinomial', 'lbfgs', False),
    ('OvR (lbfgs)',         'ovr',         'lbfgs', False),
    ('OvR (liblinear)',     'ovr',         'liblinear', False),
    ('OvR (saga)',          'ovr',         'saga', False),
    ('OvO (liblinear)',     'ovr',         'liblinear', True),
]

results_rows = []

header = f'{"Estrat\u00e9gia":<22s} {"C":<6s} {"Acur\u00e1cia":<10s} {"F1-w":<10s} {"Tempo(s)":<10s}'
print(header)
print('-' * 58)

for strat_name, multi_class, solver, is_ovo in STRATEGIES:
    for C in C_VALUES:
        if is_ovo:
            base_lr = LogisticRegression(
                C=C, max_iter=2000, solver=solver,
                multi_class=multi_class, random_state=SEED
            )
            model = OneVsOneClassifier(base_lr, n_jobs=-1)
        else:
            model = LogisticRegression(
                C=C, max_iter=2000, solver=solver,
                multi_class=multi_class, random_state=SEED,
                n_jobs=-1 if solver != 'liblinear' else 1
            )

        t0 = time.time()
        model.fit(X_train_tfidf, y_train)
        t_train = time.time() - t0

        preds = model.predict(X_val_tfidf)
        acc = accuracy_score(y_val, preds)
        f1 = f1_score(y_val, preds, average='weighted')

        results_rows.append({
            'strategy': strat_name,
            'C': C,
            'accuracy': acc,
            'f1_weighted': f1,
            'train_time': t_train,
            'model': model,
        })

        print(f'{strat_name:<22s} {C:<6.1f} {acc:<10.4f} {f1:<10.4f} {t_train:<10.2f}')

results_df = pd.DataFrame(results_rows)
print('\nExperimentos conclu\u00eddos \u2714')

Estrategia             C      Acuracia   F1-w       Tempo(s)  
----------------------------------------------------------
Multinomial (lbfgs)    0.1    0.7598     0.7516     5.32      
Multinomial (lbfgs)    1.0    0.9750     0.9750     13.99     
Multinomial (lbfgs)    10.0   0.9820     0.9820     59.93     
Multinomial (lbfgs)    100.0  0.9780     0.9780     73.18     
OvR (lbfgs)            0.1    0.7137     0.6980     5.40      
OvR (lbfgs)            1.0    0.9630     0.9630     10.12     
OvR (lbfgs)            10.0   0.9780     0.9780     12.86     
OvR (lbfgs)            100.0  0.9800     0.9800     37.14     
OvR (liblinear)        0.1    0.7137     0.6980     3.49      
OvR (liblinear)        1.0    0.9630     0.9630     6.34      
OvR (liblinear)        10.0   0.9780     0.9780     13.21     
OvR (liblinear)        100.0  0.9790     0.9790     43.11     
OvR (saga)             0.1    0.7137     0.6980     3.77      
OvR (saga)             1.0    0.9630     0.9630     3.77   

## 4. Análise por Classe (C=10.0)

In [8]:
# ─── Avaliação detalhada com C=10.0 ───────────────────────────────────
BEST_C = 10.0

detailed = []

for strat_name, multi_class, solver, is_ovo in STRATEGIES:
    if is_ovo:
        base_lr = LogisticRegression(
            C=BEST_C, max_iter=2000, solver=solver,
            multi_class=multi_class, random_state=SEED
        )
        model = OneVsOneClassifier(base_lr, n_jobs=-1)
    else:
        model = LogisticRegression(
            C=BEST_C, max_iter=2000, solver=solver,
            multi_class=multi_class, random_state=SEED,
            n_jobs=-1 if solver != 'liblinear' else 1
        )

    t0 = time.time()
    model.fit(X_train_tfidf, y_train)
    t_train = time.time() - t0

    preds = model.predict(X_val_tfidf)
    report = classification_report(y_val, preds,
                                   target_names=le.classes_,
                                   output_dict=True)

    detailed.append({
        'strategy': strat_name,
        'model': model,
        'preds': preds,
        'report': report,
        'train_time': t_train,
    })

    print(f'\n{"="*60}')
    print(f'  {strat_name} (C={BEST_C})')
    print('='*60)
    print(f'Treino: {t_train:.2f}s')
    print(classification_report(y_val, preds, target_names=le.classes_))

print('Avalia\u00e7\u00e3o detalhada conclu\u00edda \u2714')


  Multinomial (lbfgs) (C=10.0)
Treino: 135.43s
              precision    recall  f1-score   support

  Irrelevant       0.99      0.98      0.99       171
    Negative       0.99      0.98      0.99       266
     Neutral       0.99      0.97      0.98       285
    Positive       0.96      0.99      0.98       277

    accuracy                           0.98       999
   macro avg       0.98      0.98      0.98       999
weighted avg       0.98      0.98      0.98       999


  OvR (lbfgs) (C=10.0)
Treino: 22.39s
              precision    recall  f1-score   support

  Irrelevant       0.99      0.98      0.98       171
    Negative       0.98      0.98      0.98       266
     Neutral       0.99      0.97      0.98       285
    Positive       0.96      0.98      0.97       277

    accuracy                           0.98       999
   macro avg       0.98      0.98      0.98       999
weighted avg       0.98      0.98      0.98       999


  OvR (liblinear) (C=10.0)
Treino: 15.72s


## 5. Tabela Comparativa Final (C=10.0)

In [9]:
# ─── Tabela resumo ──────────────────────────────────────────────────
summary_rows = []
for entry in detailed:
    r = entry['report']
    row = {
        'Estrat\u00e9gia': entry['strategy'],
        'Tempo (s)': f"{entry['train_time']:.2f}",
        'Acur\u00e1cia': f"{r['accuracy']:.4f}",
        'F1 (weighted)': f"{r['weighted avg']['f1-score']:.4f}",
        'F1 (macro)': f"{r['macro avg']['f1-score']:.4f}",
    }
    for cls_name in le.classes_:
        row[f'F1 {cls_name}'] = f"{r[cls_name]['f1-score']:.4f}"
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
display(summary_df.style
    .set_caption('Compara\u00e7\u00e3o das Estrat\u00e9gias Multiclasse (C=10.0)')
    .background_gradient(cmap='Blues', subset=[c for c in summary_df.columns
                                                if c not in ['Estrat\u00e9gia', 'Tempo (s)']])
)

Estratégia,Tempo (s),Acurácia,F1 (weighted),F1 (macro),F1 Irrelevant,F1 Negative,F1 Neutral,F1 Positive
Multinomial (lbfgs),135.43,0.9820,0.9820,0.9829,0.9853,0.9857,0.9798,0.9767
OvR (lbfgs),22.39,0.9780,0.9779,0.9777,0.9823,0.9809,0.9712,0.9635
OvR (liblinear),15.72,0.9780,0.9779,0.9777,0.9823,0.9809,0.9712,0.9635
OvR (saga),10.07,0.9780,0.9779,0.9777,0.9823,0.9809,0.9712,0.9635
OvO (liblinear),3.89,0.9770,0.9770,0.9768,0.9758,0.9810,0.9744,0.9738


## 6. Conclusões

### Principais achados

| Estratégia | Melhor C | Acurácia | F1-weighted | Tempo (s) |
|------------|----------|-----------|-------------|----------|
| **Multinomial (lbfgs)** | 10 | **0,9820** | **0,9820** | 59,93 |
| OvR (lbfgs) | 100 | 0,9800 | 0,9800 | 37,14 |
| OvR (liblinear) | 100 | 0,9790 | 0,9790 | 43,11 |
| OvR (saga) | 100 | 0,9790 | 0,9790 | 41,66 |
| OvO (liblinear) | 100 | 0,9780 | 0,9780 | 6,82 |

### Análise

1. **Multinomial (softmax) é o mais preciso, mas drasticamente mais lento.**
   A acurácia de 0,9820 supera as demais estratégias em até 0,4 pp, mas o
   tempo de treino com C=10 é 59,93s — cerca de **5× mais lento** que OvR e
   **28× mais lento** que OvO. Isso ocorre porque o método multinomial resolve
   um problema conjunto de K classes simultaneamente, enquanto OvR e OvO
   quebram o problema em subproblemas menores e paralelizáveis.

2. **OvR com C=100 empata tecnicamente com Multinomial C=10.**
   OvR (lbfgs) com C=100 atinge 0,9800 vs 0,9820 do Multinomial C=10. A
   diferença é marginal (0,2 pp) e o OvR é muito mais rápido (37,14s vs
   59,93s). Para propósitos práticos, são equivalentes.

3. **OvO é o mais rápido, mas o menos preciso.**
   A estratégia One-vs-One com liblinear treina K×(K-1)/2 = 6 classificadores
   binários para 4 classes. Cada subproblema é pequeno (apenas 2 classes), o
   que acelera o treino (2,16s para C=10). Porém, a votação entre pares pode
   perder informações que a visão global do softmax captura.

4. **Solver saga é o mais eficiente entre os OvR.**
   Com C=10, saga treina em 9,36s vs 12,86s do lbfgs e 13,21s do liblinear,
   mantendo a mesma acurácia. O gradiente estocástico do saga converge mais
   rápido em datasets grandes como este (73k amostras).

5. **Regularização forte (C baixo) penaliza mais OvR/OvO que Multinomial.**
   Com C=0,1, Multinomial obtém 0,7598 enquanto OvR cai para 0,7137 e OvO
   para 0,6907. O softmax, por compartilhar parâmetros entre classes, é mais
   robusto à regularização forte.

### Recomendação

- **Máxima acurácia (sem preocupação com tempo)**: `multi_class='multinomial',
  solver='lbfgs', C=10`
- **Melhor custo-benefício**: `multi_class='ovr', solver='saga', C=100`
  (acurácia 0,979 em 41,66s, com suporte a L1 e L2)
- **Mínimo tempo**: `OneVsOneClassifier(LogisticRegression(solver='liblinear',
  C=100))` (0,978 em 6,82s)